In [7]:
import numpy as np
import msicpe.san as san
from plotly import express as px
import scipy.signal as signal

1.1 Signaux periodique

In [3]:
t = np.linspace(0, 4*np.pi, 2000)

signaux = {
    "x(t) = cos(3t) + sin(5t)":   np.cos(3*t) + np.sin(5*t),
    "y(t) = cos(√2 t) + sin(3t)": np.cos(np.sqrt(2)*t) + np.sin(3*t),
    "z(t) = cos(t²/2)":           np.cos(t**2/2),
}

In [12]:
for nom, s in signaux.items():
    px.line(x=t, y=s, labels={"x": "Temps (s)", "y": "Valeur"}, title=nom).show()

1.2

In [ ]:
t = np.linspace(0, 1, 500)
signaux1 = {
    "x(t) = cos(2pit)" :   np.cos(2*np.pi*t),
    "y(t) = sin(2pit)" :   np.sin(2*np.pi*t)
}


2.1

In [18]:
import pandas as pd

A, nu0 = 1, 7
T0 = 1/nu0
T = 5*T0
D = 3*T
tau0 = 1

t = np.linspace(0, D, 1000, endpoint=False)   # t ∈ [0, D[
dt = t[1] - t[0]

porte = lambda u: ((u >= 0) & (u < T)).astype(float)   # Π((u - T/2)/T)

x = A*np.cos(2*np.pi*nu0*t) * porte(t)
y = A*np.cos(2*np.pi*nu0*(t - tau0)) * porte(t - tau0)   # y(t) = x(t - τ0)

df = pd.DataFrame({
    "Temps (s)": np.concatenate([t, t]),
    "Valeur": np.concatenate([x, y]),
    "Signal": ["x(t) émis"]*len(t) + ["y(t) écho"]*len(t)
})
px.line(df, x="Temps (s)", y="Valeur", color="Signal",
        title="Impulsion émise et écho reçu").show()

In [19]:
# Inter-corrélation calculée avec msicpe
gamma, tau = san.correlate(y, x, D)

# Formule théorique (Eq. 2)
gamma_th = A**2*T/2 * (1 - np.abs(tau - tau0)/T) * np.cos(2*np.pi*nu0*(tau - tau0))
gamma_th[np.abs(tau - tau0) > T] = 0     # nulle en dehors de [τ0-T, τ0+T]

# Tracé des deux courbes superposées
df = pd.DataFrame({
    "τ (s)": np.concatenate([tau, tau]),
    "γ": np.concatenate([gamma, gamma_th]),
    "Courbe": ["γ_yx calculée"]*len(tau) + ["γ_yx théorique"]*len(tau)
})
px.line(df, x="τ (s)", y="γ", color="Courbe",
        title="Inter-corrélation γ_yx(τ)").show()

# Maximum et énergie
Ex = np.trapezoid(x**2, t)
print("Max de γ :", gamma.max(), "atteint en τ =", tau[np.argmax(gamma)])
print("Énergie Ex :", Ex, "| A²T/2 =", A**2*T/2)

Max de γ : 0.3583954356434672 atteint en τ = 1.0007142857142857
Énergie Ex : 0.3575002352227211 | A²T/2 = 0.3571428571428571


2.2

In [ ]:
dnu = 0.15*nu0          # décalage Doppler Δν
T = 5*T0
D = 5*T

t = np.linspace(0, D, 1000, endpoint=False)
porte = lambda u: ((u >= 0) & (u < T)).astype(float)

x = A*np.cos(2*np.pi*nu0*t) * porte(t)                          # émis
y = A*np.cos(2*np.pi*nu0*(t - tau0)) * porte(t - tau0)          # écho sans Doppler
w = A*np.cos(2*np.pi*(nu0 + dnu)*(t - tau0)) * porte(t - tau0)  # écho Doppler

df = pd.DataFrame({
    "Temps (s)": np.concatenate([t, t]),
    "Valeur": np.concatenate([x, w]),
    "Signal": ["x(t) émis"]*len(t) + ["w(t) écho Doppler"]*len(t)
})
px.line(df, x="Temps (s)", y="Valeur", color="Signal",
        title="Impulsion émise et écho Doppler").show()

In [ ]:
gamma_w, tau = san.correlate(w, x, D)
gamma_y, tau = san.correlate(y, x, D)


df = pd.DataFrame({
    "τ (s)": np.concatenate([tau, tau]),
    "γ": np.concatenate([gamma_y, gamma_w]),
    "Courbe": ["γ_yx (cible fixe)"]*len(tau) + ["γ_wx (Doppler)"]*len(tau)
})
px.line(df, x="τ (s)", y="γ", color="Courbe",
        title="Inter-corrélation avec et sans Doppler", range_x=[0, 2]).show()